# Phase 4 Prototype — Evidently (Drift / Quality Monitoring)

**Status: exploratory prototype, not a Phase 1 implementation item.** Per `CLAUDE.md`, Evidently is a *trigger-based addition* — it is only adopted "once you have enough production history to define meaningful reference and current windows" for drift (Phase 4). That trigger has not fired in this repo: there is no production history yet.

This notebook exists to answer one question for a stakeholder demo: **"what would drift monitoring look like?"** — not to stand up real production monitoring. It treats two existing eval runs as stand-ins for a *reference window* (an earlier/baseline model) and a *current window* (a candidate model), computes text descriptors (length, sentiment) and an LLM-judge descriptor over both, and runs a pass/fail Test Suite comparing them — the same three-layer shape as this repo's real regression gate (`compare_results.py --fail-on-regression`), but using Evidently's machinery instead.

> **This does not change the binding architecture.** `compare_results.py` remains the actual CI regression gate. Nothing here is wired into CI or any production path. If Phase 4 is later triggered for real, this notebook is a starting point, not the implementation.

## Table of contents

1. [Requirements](#1-requirements)
2. [Configuration](#2-configuration)
3. [Load two eval runs as reference / current windows](#3-load-two-eval-runs-as-reference--current-windows)
4. [Build Evidently Datasets with descriptors](#4-build-evidently-datasets-with-descriptors)
5. [Report: descriptor summary](#5-report-descriptor-summary)
6. [Test Suite: pass/fail drift + regression checks](#6-test-suite-passfail-drift--regression-checks)
7. [Export HTML report](#7-export-html-report)
8. [Feature notes: Evidently vs. this repo's compare_results.py gate](#8-feature-notes-evidently-vs-this-repos-compare_resultspy-gate)

## 1. Requirements

- `pip install -r requirements.txt` (includes `evidently`).
- Two saved eval artifacts under `notebooks/artifacts/deepeval-model-eval/` from `model_eval_deepeval_mlflow.ipynb` (or `model_eval_deepeval.ipynb`) — one to stand in as the *reference* window, one as the *current* window. Run that notebook twice (e.g. once per candidate model/endpoint) before this one if you don't have two yet.
- An LLM-judge descriptor in Section 4 needs a reachable OpenAI-compatible endpoint (reuses the same judge config pattern as the DeepEval notebooks).

In [ ]:
%pip install -q -r requirements.txt

## 2. Configuration

Pick which two saved artifacts play the role of reference and current window, and the judge endpoint for the LLM-judge descriptor (Section 4).

In [ ]:
import json
import os
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "notebooks" / "sample-prompts").exists() else Path.cwd()
assert (REPO_ROOT / "notebooks" / "sample-prompts").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "artifacts" / "deepeval-model-eval"

# ---- Reference window ("baseline") and current window ("candidate") -----
# Leave blank to auto-pick the two most recently modified results_*.json files
# (newest = current, next-newest = reference) — override with explicit
# filenames once you have specific runs you want to compare.
REFERENCE_ARTIFACT = ""   # e.g. "results_Qwen_Qwen3-0.6B.json"
CURRENT_ARTIFACT = ""     # e.g. "results_my-candidate-model.json"

if not REFERENCE_ARTIFACT or not CURRENT_ARTIFACT:
    _candidates = sorted(ARTIFACT_DIR.glob("results_*.json"), key=lambda p: p.stat().st_mtime, reverse=True) if ARTIFACT_DIR.exists() else []
    assert len(_candidates) >= 2, (
        f"Need at least 2 results_*.json files under {ARTIFACT_DIR} to compare reference vs. current. "
        "Run model_eval_deepeval_mlflow.ipynb (or model_eval_deepeval.ipynb) against two different "
        "models/endpoints first, or set REFERENCE_ARTIFACT / CURRENT_ARTIFACT explicitly above."
    )
    CURRENT_PATH, REFERENCE_PATH = _candidates[0], _candidates[1]
else:
    CURRENT_PATH = ARTIFACT_DIR / CURRENT_ARTIFACT
    REFERENCE_PATH = ARTIFACT_DIR / REFERENCE_ARTIFACT
    assert CURRENT_PATH.exists(), f"CURRENT_ARTIFACT not found: {CURRENT_PATH}"
    assert REFERENCE_PATH.exists(), f"REFERENCE_ARTIFACT not found: {REFERENCE_PATH}"

# ---- Judge endpoint for the LLM-judge descriptor (Section 4) -------------
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "gpt-4o-mini")
JUDGE_BASE_URL = os.environ.get("JUDGE_ENDPOINT", "")  # blank = use OpenAI directly via OPENAI_API_KEY
JUDGE_API_KEY = os.environ.get("JUDGE_API_KEY") or os.environ.get("OPENAI_API_KEY", "none")

print(f"Reference window: {REFERENCE_PATH.name}")
print(f"Current window:   {CURRENT_PATH.name}")
print(f"Judge:            {JUDGE_MODEL}" + (f" @ {JUDGE_BASE_URL}" if JUDGE_BASE_URL else " (OpenAI)"))

## 3. Load two eval runs as reference / current windows

Each artifact has `cases: [{input, expected_output, actual_output}, ...]` from a DeepEval run. Flatten each into a DataFrame — this is the same `(input, actual_output)` shape Evidently's text descriptors expect.

In [ ]:
import pandas as pd

def load_cases(path: Path) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        artifact = json.load(f)
    df = pd.DataFrame(artifact["cases"])
    df["model"] = artifact.get("model", path.stem)
    return df

reference_df = load_cases(REFERENCE_PATH)
current_df = load_cases(CURRENT_PATH)

print(f"Reference ({reference_df['model'].iloc[0]}): {len(reference_df)} cases")
print(f"Current   ({current_df['model'].iloc[0]}):   {len(current_df)} cases")
reference_df.head(3)

## 4. Build Evidently Datasets with descriptors

Descriptors are per-text computed values — the building block of Evidently's drift detection for text (`genai-evaluation-stack-design.md` §"Drift monitoring for LLM apps"). We use three:

- **`TextLength`** — output length; a length distribution shift is often the cheapest early signal something changed (truncation, verbosity, refusals).
- **`Sentiment`** — tone shift detector.
- **`LLMEval`** (LLM-as-judge descriptor) — a lightweight judge check, analogous to DeepEval's GEval but run through Evidently's own descriptor pipeline. This is a *separate* implementation from the `correctness`/`answer_relevancy` scores already computed by DeepEval — per `docs/metric-registry.md`'s one-metric-one-framework rule, this descriptor is **not** meant to replace or be compared numerically against those; it demonstrates Evidently's own judge mechanism, scoped to this exploratory notebook only.

In [ ]:
from evidently import Dataset, DataDefinition
from evidently.descriptors import Sentiment, TextLength, LLMEval
from evidently.llm.templates import BinaryClassificationPromptTemplate

# LLMEval needs a judge; point it at the same kind of OpenAI-compatible
# endpoint used everywhere else in this repo. If JUDGE_BASE_URL is blank this
# uses OpenAI directly via OPENAI_API_KEY, matching the DeepEval notebooks'
# "openai" backend path.
if JUDGE_API_KEY and JUDGE_API_KEY != "none":
    os.environ["OPENAI_API_KEY"] = JUDGE_API_KEY

conciseness_template = BinaryClassificationPromptTemplate(
    criteria=(
        "An ANSWER is concise if it directly addresses the QUESTION without "
        "irrelevant padding, repetition, or off-topic content."
    ),
    target_category="concise",
    non_target_category="verbose",
    pre_messages=[("system", "You are grading answer conciseness.")],
)

descriptors = [
    TextLength("actual_output", alias="Length"),
    Sentiment("actual_output", alias="Sentiment"),
    LLMEval(
        "actual_output",
        template=conciseness_template,
        provider="openai",
        model=JUDGE_MODEL,
        alias="Conciseness",
    ),
]

data_definition = DataDefinition(text_columns=["input", "actual_output"])

reference_eval = Dataset.from_pandas(reference_df, data_definition=data_definition, descriptors=descriptors)
current_eval = Dataset.from_pandas(current_df, data_definition=data_definition, descriptors=descriptors)

print("Descriptors computed. Reference sample:")
reference_eval.as_dataframe().head(3)

## 5. Report: descriptor summary

`TextEvals` summarizes descriptor distributions for both windows side by side — this is the equivalent of eyeballing a distribution shift before deciding whether it's worth a hard gate.

In [ ]:
from evidently import Report
from evidently.presets import TextEvals

report = Report([TextEvals()])
summary_eval = report.run(current_data=current_eval, reference_data=reference_eval)
summary_eval.show()

## 6. Test Suite: pass/fail drift + regression checks

This is the part that maps most directly onto this repo's real CI gate (`compare_results.py --fail-on-regression`, `CLAUDE.md`'s three-layer regression policy):

- **Layer 2 (per-metric floor)**: `Conciseness` share must stay above an absolute minimum, independent of the reference run.
- **Layer 3 (regression vs. baseline)**: `Length` must not drift beyond a tolerance relative to the reference window.

Evidently's `tests=[...]` argument attaches pass/fail conditions directly to a metric; `report.run()` returns both the descriptor values and each test's status.

In [ ]:
from evidently.metrics import MeanValue
from evidently.tests import gte, lte

# Layer 2: absolute floor. Adjust to a real calibrated threshold before this
# is ever load-bearing — 0.5 here is illustrative only.
CONCISENESS_FLOOR = 0.5

# Layer 3: regression tolerance vs. reference mean length (relative, %).
LENGTH_DRIFT_TOLERANCE = 0.30

ref_mean_length = reference_eval.as_dataframe()["Length"].mean()
length_lower = ref_mean_length * (1 - LENGTH_DRIFT_TOLERANCE)
length_upper = ref_mean_length * (1 + LENGTH_DRIFT_TOLERANCE)

gate_report = Report([
    MeanValue(column="Conciseness", tests=[gte(CONCISENESS_FLOOR)]),
    MeanValue(column="Length", tests=[gte(length_lower), lte(length_upper)]),
])
gate_eval = gate_report.run(current_data=current_eval, reference_data=reference_eval)
gate_eval.show()

print("\nProgrammatic pass/fail (what a CI job would branch on):")
all_passed = True
for test_result in gate_eval.tests:
    status = test_result.result.status
    all_passed &= (status == "SUCCESS")
    print(f"  [{status}] {test_result.name}")

print(f"\nGate result: {'PASS' if all_passed else 'FAIL'}")

## 7. Export HTML report

Evidently reports render to standalone HTML/JSON offline (air-gap-friendly, same principle as this repo's other artifacts) — save one for sharing with a stakeholder who doesn't have the notebook running.

In [ ]:
out_dir = REPO_ROOT / "notebooks" / "artifacts" / "evidently-drift-prototype"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"drift_{REFERENCE_PATH.stem}_vs_{CURRENT_PATH.stem}.html"

summary_eval.save_html(str(out_path))
print(f"Saved -> {out_path}")

## 8. Feature notes: Evidently vs. this repo's `compare_results.py` gate

- **Same three-layer shape, different machinery.** `compare_results.py --fail-on-regression` already implements hard constraints / floors / regression-vs-baseline (`CLAUDE.md`'s Regression Gate Policy). Section 6 above reproduces that shape using Evidently's `tests=[gte(...), lte(...)]` API instead of hand-written comparison logic — mechanically similar, not a capability compare_results.py lacks.
- **What Evidently adds that compare_results.py doesn't have today**: built-in *distributional* drift tests (PSI, Kolmogorov-Smirnov) for descriptors over many samples, not just a two-run point comparison. That only becomes meaningful with real production volume — which is exactly the Phase 4 trigger condition in `CLAUDE.md`, not something two golden-set runs can validate.
- **The LLMEval descriptor is a second, unregistered judge implementation.** It is deliberately scoped to this notebook — per the metric-registry rule, it must not be treated as comparable to DeepEval's `correctness`/`answer_relevancy` scores, and it is not added to `docs/metric-registry.md`.
- **Reference/current windows here are two golden-set runs, not real time-windowed production traffic** — a legitimate stand-in for demoing the mechanism, but not evidence that Evidently's drift tests are calibrated or useful yet. Real reference/current windows need enough production history to be statistically meaningful.
- **Air-gap**: Evidently reports render to HTML/JSON offline with no mandatory telemetry for evals (per `genai-evaluation-stack-design.md` §B.3) — consistent with this repo's air-gap requirements, though `PHOENIX_TELEMETRY_ENABLED`-style verification against the exact installed version is still worth doing before a real deployment.

**Bottom line**: this confirms Evidently's Report/Test Suite API can express the same three-layer gate shape this repo already has, using a different toolkit — useful mainly as validation that Evidently's API is a plausible backend for drift monitoring, once real production windows exist. It is not evidence to trigger Phase 4 early; the entry criterion (real production history) still governs.